In [0]:
"""
03_production_kpis.py

Streaming Production KPIs.

Computes production KPIs from Work Orders,
Executions and Serial Number Events.

Inputs:
    work_order_events
    execution_events
    serial_number_events

Output:
    production_kpis

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
)


# ============================================================
# Production KPIs
# ============================================================

@dlt.table(
    name="production_kpis_batch",
    comment="Production KPIs aggregated from work orders, executions and serial numbers.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect_or_drop(
    "valid_product_code",
    "product_code IS NOT NULL",
)

@dlt.expect_or_drop(
    "valid_shift",
    "planned_shift IS NOT NULL",
)

@dlt.expect(
    "positive_work_orders",
    "work_orders_created > 0",
)

def production_kpis():

    work_orders = dlt.read("work_order_events")
    executions = dlt.read("execution_events")
    serial_numbers = dlt.read("serial_number_events")

    return (

        work_orders.alias("wo")

        .join(
            executions.alias("ex"),
            on="work_order_id",
            how="left",
        )

        .join(
            serial_numbers.alias("sn"),
            on="execution_id",
            how="left",
        )

        .groupBy(

            col("wo.plant_code"),

            col("wo.product_code"),

            col("wo.product_name"),

            col("wo.family"),

            col("wo.planned_shift"),

        )

        .agg(

            count(
                "wo.work_order_id"
            ).alias(
                "work_orders_created"
            ),

            count(
                "ex.execution_id"
            ).alias(
                "executions_started"
            ),

            count(
                "sn.serial_number"
            ).alias(
                "products_started"
            ),

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp(),

        )

    )